In [1]:
import pandas as pd
import numpy as np
import os
import json
from collections import defaultdict
# source ~/anaconda3/bin/activate

#### 1. Analysis of PreMOTA off-target Prediction Results

In [ ]:
data_predict_result_path = "/home/datahouse1/liujin/HetSia-SafeNet/Data/investigational_drugs/"
predict_dict_path = os.path.join(data_predict_result_path, 'drugs_offtarget_predict_result.json') # {smiles1:{target1: [pK, pAC50], target2: [pK, pAC50]...},smiles2:{target1: [pK, pAC50], target2: [pK, pAC50]...}}
data_path = os.path.join(data_predict_result_path, 'drugs_info.csv') # Drug,smiles_uncanical,smiles,label

data_df = pd.read_csv(data_path) # canonical_smiles、label
adr_smiles_list = data_df['canonical_smiles'].tolist()
print("data_df.shape: ", data_df.shape)

with open(predict_dict_path, 'r') as f:
    predict_dict = json.load(f)

print("len(predict_dict): ", len(predict_dict))

predict_dict_new = defaultdict(dict)
for key,value in predict_dict.items():
    if key in adr_smiles_list:
        for k,v in value.items():
            value1 = v[0] # pK
            value2 = v[1] # pAC50
            value1_uM = 1e6*10**(-value1)
            value2_uM = 1e6*10**(-value2)
            values_new = (value1_uM+value2_uM)/2
            predict_dict_new[key][k] = value2_uM
            # predict_dict_new[key][k] = value2


aff_df = pd.DataFrame(predict_dict_new).T
aff_df.index.name = 'smiles'
aff_df.reset_index(inplace=True)

print("df.shape: ", aff_df.shape)
print("parse done!!!")

aff_df.to_csv(os.path.join(data_predict_result_path, 'drug_offtarget_predict.csv'), index=False)

data_df.shape:  (548, 4)
len(predict_dict):  548
df.shape:  (548, 195)
parse done!!!


In [ ]:
aff_df = pd.read_csv(os.path.join(data_predict_result_path, 'drug_offtarget_predict.csv'))
feature_list = aff_df.columns.tolist()[1:]
print(len(feature_list))
print("df.shape: ", aff_df.shape)
aff_df.head()

194
df.shape:  (548, 195)


,smiles,P35414,P24530,P30518,P35346,Q9Y271,P34998,P30874,P34995,P08173,...,P23975,P31645,Q01959,P31652,P23977,Q9GZV3,P28572,P30536,P01375,Q99720
0,CN(C(=O)n1cnc(-c2ccc[n+]([O-])c2)c1)C1CCCCC1,6.811046,32.785027,2.095033,26.178049,1.982846,2.380034,2.231998,3.894949,0.232969,...,2.232359,1.492522,1.306164,0.486064,0.963738,6.654002,1.098264,0.088721,8.684580,2.506729
1,Cc1c(C(=O)NN2CCCCC2)nn(-c2ccc(Cl)cc2Cl)c1-c1cc...,1.208370,1.434329,0.394548,0.647414,12.435128,0.172672,0.241946,0.055816,1.999720,...,1.607715,1.804178,2.929053,0.808884,2.465575,4.230121,0.302039,4.820622,0.793654,1.396152
2,CNCc1ccc(-c2[nH]c3cc(F)cc4c3c2CCNC4=O)cc1,0.209194,13.600039,3.594262,0.277033,7.105957,2.541362,0.387392,2.617673,1.446414,...,0.544643,0.407671,1.239731,0.693596,0.676614,9.775941,0.587317,793.770000,0.660926,0.180747
3,O=C(O)CCCc1cc2cc(Cl)ccc2n1S(=O)(=O)c1ccc2ncsc2c1,0.812621,16.576721,0.592523,0.267268,0.070923,0.830676,0.075474,0.158067,1.323974,...,10.559561,1.472194,11.508607,0.078645,1.750037,2.331614,0.339266,0.012073,5.644141,3.825350
4,CCN(CCO)CCCC(C)Nc1ccnc2cc(Cl)ccc12,17.688156,23.502020,5.633956,4.127267,34.082823,1.224859,1.397898,2.142092,0.820567,...,61.521948,2.851951,8.582000,1.794703,5.572406,16.266181,0.695090,5.428582,28.923019,0.424426


#### 2. Analysis of MotifAttnNet Prediction Results

In [ ]:
dose_list = ["5mg","10mg","25mg","50mg","75mg","100mg","125mg","150mg","200mg","250mg","300mg","350mg"]
dose_predict_result = "/home/datahouse1/liujin/HetSia-SafeNet/Data/investigational_drugs/dose_study/"
aff_dff = pd.read_csv(os.path.join(data_predict_result_path, 'drug_offtarget_predict.csv'))
aff_dff_o = pd.read_csv(os.path.join(data_predict_result_path, 'drugs_info.csv'))
drug_name_dict = dict(zip(aff_dff_o['canonical_smiles'], aff_dff_o['drug_name']))

# drug_cmax_predict_10mg.csv
for dose in dose_list:
    dose_df_path = os.path.join(dose_predict_result,f"drug_cmax_predict_{dose}.csv") # Drug|smiles|cmax_free(uM)
    dose_df = dose_df[dose_df['cmax_free(uM)']>0]
    # dose_df['cmax_free(uM)'] = dose_df['cmax_free(uM)'].apply(lambda x: np.log10(x))

    drug_dose_dict = dict(zip(dose_df['smiles'],dose_df['cmax_free(uM)']))
    print("药物剂量-Cmax字典长度：",len(drug_dose_dict))
    aff_df = aff_dff.copy()
    feature_list = aff_dff.columns.tolist()[1:]
    print("off-target length: ",len(feature_list))
    aff_df['Drug'] = aff_df['smiles'].map(drug_name_dict)
    aff_df['cmax_free(uM)'] = aff_df['smiles'].map(drug_dose_dict)
 
    feature_list.insert(0, 'Drug')
    feature_list.insert(1, 'smiles')
    feature_list.insert(2, 'cmax_free(uM)')

    aff_df_choose = aff_df[feature_list]
    # Take log10 from the data after the cmax_free(uM) column
    aff_df_choose.iloc[:,2:] = np.log10(aff_df_choose.iloc[:,2:])
    print("aff_df_choose.shape: ", aff_df_choose.shape)
    
    new_columns = {aff_df_choose.columns[i] + '*Cmax': aff_df_choose[aff_df_choose.columns[i]] * aff_df_choose['cmax_free(uM)'] for i in range(3,len(aff_df_choose.columns))}
    aff_df_choose_cmax = pd.concat([aff_df_choose, pd.DataFrame(new_columns)], axis=1)
    print("aff_df_choose_cmax.shape: ", aff_df_choose_cmax.shape)
    aff_df_choose_cmax.dropna(subset=['cmax_free(uM)'], inplace=True)
    print("aff_df_choose_cmax.shape: ", aff_df_choose_cmax.shape)
    aff_df_choose_cmax.to_csv(f"/home/datahouse1/liujin/HetSia-SafeNet/Data/investigational_drugs/dose_study_offcmax/predict_X_{dose}.csv",index=False)
    

药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choose_cmax.shape:  (548, 391)
aff_df_choose_cmax.shape:  (547, 391)
药物剂量-Cmax字典长度： 547
off-target length:  194
aff_df_choose.shape:  (548, 197)
aff_df_choos

In [2]:
import pandas as pd
df1 = pd.read_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/investigational_drugs/dose_study_offcmax/predict_X_10mg.csv")
df2 = pd.read_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/AC50_Cmax/test_X.csv")
print(df1.shape)
print(df2.shape)
df1.head()

(547, 391)
(921, 391)


,Drug,smiles,cmax_free(uM),P35414,P24530,P30518,P35346,Q9Y271,P34998,P30874,...,P23975*Cmax,P31645*Cmax,Q01959*Cmax,P31652*Cmax,P23977*Cmax,Q9GZV3*Cmax,P28572*Cmax,P30536*Cmax,P01375*Cmax,Q99720*Cmax
0,BIA 10-2474,CN(C(=O)n1cnc(-c2ccc[n+]([O-])c2)c1)C1CCCCC1,-0.881777,0.833214,1.515676,0.321191,1.417937,0.297289,0.376583,0.348694,...,-0.307532,-0.153359,-0.102284,0.276266,0.014144,-0.725775,-0.035894,0.927607,-0.827767,-0.351924
1,Acomplia,Cc1c(C(=O)NN2CCCCC2)nn(-c2ccc(Cl)cc2Cl)c1-c1cc...,-3.539713,0.082200,0.156649,-0.403900,-0.188818,1.094650,-0.762777,-0.616282,...,-0.729921,-0.907156,-1.652081,0.326055,-1.387278,-2.217109,1.840430,-2.417989,0.355277,-0.513020
2,Rubraca,CNCc1ccc(-c2[nH]c3cc(F)cc4c3c2CCNC4=O)cc1,-1.893371,-0.679451,1.133540,0.555610,-0.557468,0.851623,0.405066,-0.411849,...,0.499639,0.737829,-0.176704,0.300844,0.321228,-1.874738,0.437610,-5.490198,0.340518,1.406641
3,lanifibranor,O=C(O)CCCc1cc2cc(Cl)ccc2n1S(=O)(=O)c1ccc2ncsc2c1,-1.612269,-0.090112,1.219499,-0.227295,-0.573052,-1.149215,-0.080568,-1.122204,...,-1.650392,-0.270805,-1.710654,1.780479,-0.391858,-0.592761,0.756895,3.092639,-1.211778,-0.939423
4,hydroxychloroquine,CCN(CCO)CCCC(C)Nc1ccnc2cc(Cl)ccc12,-1.940189,1.247683,1.371105,0.750813,0.615663,1.532536,0.088086,0.145475,...,-3.471057,-0.883062,-1.811338,-0.492794,-1.447464,-2.350123,0.306470,-1.425430,-2.835089,0.722134
